In [1]:
import numpy as np
import cv2
import os
import pickle
import random
import shutil
import sys
import time
import zipfile
from matplotlib.collections import LineCollection
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
import random
import torch
import pandas

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))  # si el notebook está dentro de /notebooks


In [ ]:
from domain.Task import Task
from domain.Stroke import Stroke
from domain.RepresentationType import RepresentationType
from utils.CustomMorphOps import bresenham_line, normalize, fit_into_normalized_canvas

In [ ]:
def _rep_enhanced_stroke(self, min_thickness = 2, max_thickness = 10, min_dark_factor = 0.7, max_dark_factor = 0.99):
        final_w = int(self.max_vals['x_surface'] - self.min_vals['x_surface'])
        final_h = int(self.max_vals['y_surface'] - self.min_vals['y_surface'])
        if final_h == 0 or final_w == 0:
            print(f"[DEBUG] Subject {self.subject_id} task {self.task_number} -> H: {final_h}, W: {final_w}")
        canvas = np.ones((final_h, final_w), dtype=np.float32)
        for letters_set in self.letters_sets_list:
            for stroke in letters_set.strokes_list:
                stroke_x_list = stroke.get_x_coordinates_list()
                stroke_y_list = stroke.get_y_coordinates_list()
                normalized_x = [x - self.min_vals['x_surface'] for x in stroke_x_list]
                normalized_y = [y - self.min_vals['y_surface'] for y in stroke_y_list]
                altitudes = stroke.getAltitudes()
                normalized_altitudes = normalize(altitudes)
                pressures = stroke.getPressures()
                normalized_pressures = normalize(pressures)
                for i in range(len(stroke_x_list) -1):
                    darkening_factor = min_dark_factor + (max_dark_factor - min_dark_factor) * (1 - normalized_pressures[i])
                    thickness_factor = min_thickness + (max_thickness - min_thickness) * (1 - normalized_altitudes[i])
                    pixels = bresenham_line(
                        normalized_x[i],
                        normalized_y[i],
                        normalized_x[i+1],
                        normalized_y[i+1],
                        height=final_h,
                        width=final_w,
                        thickness=int(thickness_factor),
                        )
                    for y, x in pixels:
                        canvas[y, x] *= darkening_factor

        flip_img = cv2.flip(canvas, 0)
        negative_img = 1.0 - flip_img
        return negative_img

In [ ]:
def read_csv(task_file_path=None, subject_id=1, task_num=2):
    task_strokes_list = []
    all_coords = []
    print(task_file_path)
    if os.path.exists(task_file_path):
        with open(task_file_path, encoding="utf-8") as task_file:
            # Se salta la primera línea.
            max_x = 0
            max_y = 0
            task_file.readline()
            from_on_air = True
            while True:
                line = task_file.readline()
                if not line:
                    break
                x = int(line.split()[1])
                y = int(line.split()[0])
                if x > max_x:
                    max_x = x
                if y > max_y:
                    max_y = y
                coordinate = (
                    x,
                    y,
                    int(line.split()[2]),
                    int(line.split()[3]),
                    int(line.split()[4]),
                    int(line.split()[5]),
                    int(line.split()[6]),
                )
                all_coords.append(coordinate)
                # Si la coordenada está sobre el papel.
                if line.split()[3] == "1":
                    if from_on_air:
                        task_strokes_list.append(Stroke(coordinate))
                        from_on_air = False
                    else:
                        task_strokes_list[-1].append(coordinate)
                else:
                    from_on_air = True
    else:
        print(f"Archivo no encontrado: {task_file_path}, se omite.")
    if task_strokes_list:  # Solo si hay trazos

        print("Num strokes:", len(task_strokes_list))

        lengths = [len(s) for s in task_strokes_list]

        print("Primeras longitudes:", lengths[:20])

        new_task = Task(subject_id, task_num, task_strokes_list, all_coords, 0, rep_type=RepresentationType.SIMPLE_STROKE)
        img = _rep_enhanced_stroke()
        write_img = (img * 255).astype(np.uint8)
        cv2.imwrite("EMOTHAWWWW.png", write_img)
        return new_task
    else:
        print(f"Tarea vacía para Sujeto {subject_id}, Tarea {task_num}, se omite.")



In [7]:
#Read corpus pahaw xslx
pahaw_file_path = os.path.join("../PaHaW", "PaHaW_files", "corpus_PaHaW.xlsx")
#pahaw_dataframe = pandas.read_excel(pahaw_file_path)
#
##random id and task
#subject_id = 5
#task_num = 6
#
##Read all csvs
#task_file_path_start = "/home/dcorredor/github/Proyecto-ParkinsonDisease/NEW/PaHaW/PaHaW_public"
#task_file_path_end = "_1.svc"
#task_file_path_mid = os.path.join(
#    f"{subject_id:05d}", f"{subject_id:05d}__{task_num}"
#)
#final_task_file_path = os.path.join(
#    task_file_path_start, task_file_path_mid + task_file_path_end
#)

final_task_file_path = "/home/dcorredor/Dani/EMOTHaw/archive/DataEmothaw/Collection1/user00001/session00001/u00001s00001_hw00007.svc"

random_task = read_csv(final_task_file_path)

/home/dcorredor/Dani/EMOTHaw/archive/DataEmothaw/Collection1/user00001/session00001/u00001s00001_hw00007.svc
Num strokes: 34
Primeras longitudes: [59, 48, 130, 10, 42, 13, 13, 6, 220, 80, 50, 100, 8, 23, 14, 25, 68, 9, 35, 71]
